# Artistic Telephone: try, compare, choose, pass

Run sections 1 and 2 once. Each round, run 3 and 4, then repeat **5 and 6** to iterate. Use **7** to compare and select, then **8** to save the selected image for passing.

Every completed attempt stays private. Only your chosen attempt becomes a handoff. Start in offline practice mode: the labelled diagram makes no API calls and does not respond to your prompt. Agree a time and spending budget with the instructor for live attempts.

Keep all `telephone_*.py` helpers beside this notebook. See [setup](README.md) and [game rules](../notes/artistic_telephone.md). Works online or in person.

## 1. Set up once
Use the same session name as everyone in class. Keep group and seat fixed. Saved attempts survive a kernel restart. Use recovery below if a prediction is pending.

In [ ]:
from pathlib import Path
from copy import deepcopy
from getpass import getpass
import json
import ipywidgets as widgets
from IPython.display import display, Image, FileLink
from telephone_helpers import MODELS, route, session_root, uploaded_image, history, export_handoff, error_message
from telephone_attempts import Attempts

SESSION = "h26_w5"
GROUP = "A"
SEAT = 1
LIVE = False

root = session_root(SESSION, GROUP, SEAT)
attempts = Attempts(root)
client = None
model_choice = widgets.Dropdown(options=list(MODELS), description="Model:")
display(model_choice)
print("Private work:", root)
if (root / "pending.json").exists():
    print("Pending prediction recorded. Use recovery below before generating.")

## 2. Connect your account in live mode
Paste your API token only into the hidden input, never into code. Restart the kernel when finished. The HTTP timeout is not a limit on total generation time.

In [ ]:
if LIVE:
    import replicate
    client = replicate.Client(api_token=getpass("Replicate API token: "), timeout=120)
else:
    print("Offline practice. No token needed.")

## 3. Start a round
Change ROUND and run this once per round. It clears the previous incoming image and prompt, but keeps saved attempts. Everyone starts a separate chain in round 1.

In [ ]:
ROUND = 1
assignment = route(GROUP, SEAT, ROUND)
incoming_bytes = None
source_filename = None
incoming_upload = widgets.FileUpload(accept=".png", multiple=False)
prompt_box = widgets.Textarea(value="", description="Prompt:", layout=widgets.Layout(width="95%", height="140px"))
observation_box = widgets.Textarea(value="", description="Observe:", layout=widgets.Layout(width="95%", height="70px"))
print("Group:", assignment["folder"], "| Round:", ROUND)
print("Receive:", assignment["incoming"] or "Nothing: invent a starting scene")
print("Pass:", assignment["outgoing"], "| Next seat:", assignment["send_to"])
if assignment["incoming"]:
    display(incoming_upload)
display(prompt_box, observation_box)

## 4. Inspect your incoming image
In rounds 2 to 5, download the assigned PNG from your group's SWITCHdrive folder, upload it above, then run this cell. Open only your assigned image. Describe what you see without looking for its hidden prompt. The incoming image is not sent to Replicate.

If upload widgets fail, set INCOMING_PATH to the downloaded file on the Jupyter server. Prompt widgets require ipywidgets support.

In [ ]:
INCOMING_PATH = ""
incoming_bytes = None
source_filename = None
if assignment["incoming"]:
    value = incoming_upload.value
    if INCOMING_PATH:
        local = Path(INCOMING_PATH)
        value = [{"name": local.name, "content": local.read_bytes()}]
    try:
        incoming_bytes = uploaded_image(value, assignment["incoming"])
        source_filename = assignment["incoming"]
        display(Image(data=incoming_bytes, width=420))
    except ValueError as error:
        print(error)
else:
    print("Round 1: use your group's theme.")

## 5. Generate an attempt
Edit the prompt above, then run this cell. In live mode type GENERATE for one potentially paid prediction. Always run section 6 next before trying again.

The prompt, model and settings are captured now. Editing the prompt afterwards will not relabel an earlier image.

In [ ]:
try:
    if (root / "pending.json").exists() and attempts.pending is None:
        raise ValueError("Recover the recorded prediction before generating. See recovery below.")
    if attempts.pending is not None:
        raise ValueError("An attempt is pending. Run section 6 to finish saving it.")
    config = deepcopy(MODELS[model_choice.value])
    context = {
        **assignment, "session": SESSION, "mode": "live" if LIVE else "practice",
        "prompt": prompt_box.value, "observation": observation_box.value,
        "model": config["identifier"], "settings": config["input"],
        "source_filename": source_filename,
    }
    if not LIVE or input("Type GENERATE for one paid attempt: ").strip() == "GENERATE":
        attempts.generate(context, client, incoming_bytes)
        print("Attempt started. Run section 6 now.")
    else:
        print("Cancelled. No new prediction requested.")
except ValueError as error:
    print(error)
except Exception as error:
    print(error_message(error))

## 6. Save and preview this attempt privately
Run after every generation. If waiting or downloading fails, rerun **this cell**, which does not create another prediction. Successful attempts are saved in separate folders.

To iterate: inspect the image, revise the prompt above, then repeat **5 and 6** within the round's time and spending budget.

In [ ]:
try:
    saved_run = attempts.save()
    display(Image(filename=str(saved_run / "image.png"), width=420))
    print("Saved privately:", saved_run)
except ValueError as error:
    print(error)
except Exception as error:
    print("Could not finish saving. Retry this cell before generating again.")
    print(error_message(error))

## 7. Compare and choose
Run to refresh attempts for this round and mode. Choose any attempt in the dropdown, including an earlier one. The most recent is not automatically passed.

Rerunning this cell resets selection to the first option, so check your choice before exporting.

In [ ]:
current_mode = "live" if LIVE else "practice"
records = [r for r in history(root) if r["round"] == assignment["round"] and r["mode"] == current_mode]
choices = []
for number, record in enumerate(records, 1):
    run = Path(record["run_path"])
    print("Attempt", number, "|", record["prompt"])
    display(Image(filename=str(run / "image.png"), width=260))
    choices.append((f"Attempt {number}", str(run)))
selection = widgets.Dropdown(options=choices, description="Pass:")
display(selection)
if not records:
    print("No saved attempts for this round yet.")

## 8. Save selected image for passing
Choose above, then run this cell. It uses that attempt's original prompt and settings, not the current prompt box. Download the PNG through its link and upload to your group's SWITCHdrive folder without renaming.

A small hidden record is added in the PNG's pixels for the instructor's reveal. It is not ordinary PNG metadata, but is also not encryption. Do not decode others' prompts during play.

**Pass the original PNG.** Screenshots, resizing and JPEG conversion can destroy the record. Keep every round's image in the shared folder. An exported round cannot silently be replaced.

In [ ]:
if selection.value is None:
    print("Choose a saved attempt first.")
else:
    selected_run = Path(selection.value)
    selected_record = json.loads((selected_run / "record.json").read_text(encoding="utf-8"))
    if selected_record["round"] != assignment["round"] or selected_record["mode"] != ("live" if LIVE else "practice"):
        print("Selection is from another round or mode. Refresh section 7.")
    else:
        try:
            handoff = export_handoff(root, selected_run)
            display(FileLink(str(handoff)))
            print("Upload this PNG to", assignment["folder"], "without renaming or editing.")
        except (ValueError, FileExistsError) as error:
            print(error)

## Recovery and next round
For the next round, update ROUND and rerun 3 and 4. All previous attempts remain saved.

After restarting the kernel, use the same SESSION, GROUP and SEAT, reconnect your account, then set RECOVER to True below if a pending prediction was recorded. Run section 6 afterwards. This fetches the existing prediction instead of generating again.

If Replicate confirms it failed or was canceled, set CLEAR_FAILED to True below before a new attempt. Successful or running predictions cannot be cleared this way. If a request failed before returning an ID, check Replicate before retrying.

In [ ]:
RECOVER = False
CLEAR_FAILED = False
if RECOVER and client is not None:
    attempts.resume(client)
    print("Recovered prediction. Run section 6.")
if CLEAR_FAILED:
    attempts.discard_failed()
    print("Confirmed failed/canceled prediction cleared. You may try again.")

## Reveal and reflection
The instructor collects all original PNGs from each group folder and opens notebook 03. No private prompt-log upload is needed to reconstruct selected chains.

Discuss what stayed consistent, what drifted, and why you selected your attempt. Keep private attempts for your own comparisons and later evaluation. Never share API tokens.